# StatsBomb Data Analysis

This notebook documents the required basic data analysis for the semestral project. It works with the three linked datasets prepared for the MongoDB cluster: `matches`, `players`, and `events`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve().parents[1] if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / 'Data' if (PROJECT_ROOT / 'Data').exists() else PROJECT_ROOT
PROCESSED_DIR = DATA_DIR / 'processed'
ANALYSIS_DIR = DATA_DIR / 'analysis'
PLOTS_DIR = ANALYSIS_DIR / 'plots'
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid')

matches = pd.read_csv(PROCESSED_DIR / 'matches.csv')
players = pd.read_csv(PROCESSED_DIR / 'players.csv')
events = pd.read_csv(PROCESSED_DIR / 'events.csv')


## Dataset overview

In [ ]:
overview = pd.DataFrame([
    {'dataset': 'matches', 'rows': len(matches), 'columns': len(matches.columns), 'total_missing_values': int(matches.isna().sum().sum())},
    {'dataset': 'players', 'rows': len(players), 'columns': len(players.columns), 'total_missing_values': int(players.isna().sum().sum())},
    {'dataset': 'events', 'rows': len(events), 'columns': len(events.columns), 'total_missing_values': int(events.isna().sum().sum())},
]).sort_values('rows', ascending=False)
overview

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

sns.barplot(data=overview, x='dataset', y='rows', hue='dataset', dodge=False, ax=axes[0])
axes[0].set_title('Dataset sizes (linear scale)')
legend = axes[0].get_legend()
if legend is not None:
    legend.remove()
for container in axes[0].containers:
    axes[0].bar_label(container, labels=[f"{int(bar.get_height()):,}" for bar in container], padding=3, fontsize=9)

sns.barplot(data=overview, x='dataset', y='rows', hue='dataset', dodge=False, ax=axes[1])
axes[1].set_title('Dataset sizes (log scale)')
axes[1].set_yscale('log')
legend = axes[1].get_legend()
if legend is not None:
    legend.remove()
for container in axes[1].containers:
    axes[1].bar_label(container, labels=[f"{int(bar.get_height()):,}" for bar in container], padding=3, fontsize=9)

plt.tight_layout()
plt.show()

## Missing values

In [ ]:
def missing_summary(name, df):
    out = pd.DataFrame({'column': df.columns, 'missing_count': df.isna().sum().values})
    out['missing_pct'] = (out['missing_count'] / max(len(df), 1) * 100).round(2)
    out['dataset'] = name
    return out.sort_values(['missing_count', 'column'], ascending=[False, True]).reset_index(drop=True)

missing_all = pd.concat([
    missing_summary('matches', matches),
    missing_summary('players', players),
    missing_summary('events', events),
], ignore_index=True)

missing_all.head(15)

## Numeric statistics

In [ ]:
events.select_dtypes(include='number').describe().T.head(12)

## Event distribution

In [ ]:
event_types = events['event_type_name'].value_counts().head(10).rename_axis('event_type_name').reset_index(name='count')
event_types

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=event_types, x='count', y='event_type_name', hue='event_type_name', dodge=False, ax=ax)
ax.set_title('Top 10 event types')
ax.legend_.remove()
plt.tight_layout()
plt.show()

## Shot outcomes

In [ ]:
shot_outcomes = (
    events.loc[events['event_type_name'] == 'Shot', 'shot_outcome_name']
    .fillna('Unknown')
    .value_counts()
    .head(8)
    .rename_axis('shot_outcome_name')
    .reset_index(name='count')
)
shot_outcomes

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=shot_outcomes, x='count', y='shot_outcome_name', hue='shot_outcome_name', dodge=False, ax=ax)
ax.set_title('Shot outcomes')
ax.legend_.remove()
plt.tight_layout()
plt.show()

## Player countries

In [ ]:
countries = players['country'].fillna('Unknown').value_counts().head(10).rename_axis('country').reset_index(name='count')
countries

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=countries, x='count', y='country', hue='country', dodge=False, ax=ax)
ax.set_title('Top 10 player countries')
ax.legend_.remove()
plt.tight_layout()
plt.show()